In [1]:
import pandas as pd
import re

# Load the CSV file to inspect its contents
file_path = 'tweets.csv'
data = pd.read_csv(file_path)

In [7]:
data['created_at'] = pd.to_datetime(data['created_at'], errors='coerce')

In [8]:
def extract_match_info(text, created_at):
    match_info = {}
    
    # Extract date (use the created_at timestamp if no date in text)
    date_match = re.search(r'\d{4}-\d{2}-\d{2}', text)
    match_info['date'] = pd.to_datetime(created_at)
    
    # Extract teams
    teams = re.findall(r'(?:[A-Z][a-z]+\s?)+\d*', text)
    if len(teams) >= 2:
        match_info['home_team'] = teams[0].split()[0]
        match_info['away_team'] = teams[1].split()[0]
    else:
        match_info['home_team'], match_info['away_team'] = None, None
    
    # Extract xG values
    xg_values = re.findall(r'xG:\s*([\d\.]+)\s*-\s*([\d\.]+)', text)
    if xg_values:
        match_info['home_xg'], match_info['away_xg'] = map(float, xg_values[0])
    else:
        match_info['home_xg'], match_info['away_xg'] = None, None
    
    # Extract xThreat values
    xthreat_values = re.findall(r'xThreat:\s*([\d\.]+)\s*-\s*([\d\.]+)', text)
    if xthreat_values:
        match_info['home_xthreat'], match_info['away_xthreat'] = map(float, xthreat_values[0])
    else:
        match_info['home_xthreat'], match_info['away_xthreat'] = None, None
    
    return match_info

# Extract match data from each tweet
match_data = []
for _, row in data.iterrows():
    match_details = extract_match_info(row['text'], row['created_at'])
    match_data.append(match_details)

# Convert the extracted data into a DataFrame
match_df = pd.DataFrame(match_data, columns=['date', 'home_team', 'away_team', 'home_xg', 'away_xg', 'home_xthreat', 'away_xthreat'])

# Display the resulting DataFrame
match_df.head()

,date,home_team,away_team,home_xg,away_xg,home_xthreat,away_xthreat
0,2024-08-10 15:59:46+00:00,Manuel,Man,NaN,NaN,NaN,NaN
1,2024-08-10 15:59:42+00:00,Man,Man,NaN,NaN,NaN,NaN
2,2024-08-10 15:59:30+00:00,Man,Man,NaN,NaN,NaN,NaN
3,2024-08-10 15:59:19+00:00,Man,Man,0.5,1.21,1.09,0.77
4,2024-07-15 04:12:12+00:00,Davinson,Argentina,NaN,NaN,NaN,NaN


In [18]:
df = match_df.dropna(subset=['home_xg'])
df = df[df['home_team'] != 'Threat']
df = df[df['away_team'] != 'Threat']

In [20]:
# Step 1: Create total_xg_for_match column
df['total_xg_for_match'] = df['home_xg'] + df['away_xg']

# Step 2: Sort the data by date
df = df.sort_values(by='date')

# Step 3: Create a function to get the previous xG or xThreat for a team
def get_previous_stat(df, team, current_date, stat_column, home_stat_column, away_stat_column):
    previous_games = df[((df['home_team'] == team) | (df['away_team'] == team)) & (df['date'] < current_date)]
    if not previous_games.empty:
        last_game = previous_games.iloc[-1]
        if last_game['home_team'] == team:
            return last_game[home_stat_column]
        else:
            return last_game[away_stat_column]
    return None

# Step 4: Apply the function to create the previous xG and xThreat columns
df['home_team_prev_xg'] = df.apply(lambda row: get_previous_stat(df, row['home_team'], row['date'], 'xg', 'home_xg', 'away_xg'), axis=1)
df['away_team_prev_xg'] = df.apply(lambda row: get_previous_stat(df, row['away_team'], row['date'], 'xg', 'home_xg', 'away_xg'), axis=1)
df['home_team_prev_xthreat'] = df.apply(lambda row: get_previous_stat(df, row['home_team'], row['date'], 'xthreat', 'home_xthreat', 'away_xthreat'), axis=1)
df['away_team_prev_xthreat'] = df.apply(lambda row: get_previous_stat(df, row['away_team'], row['date'], 'xthreat', 'home_xthreat', 'away_xthreat'), axis=1)

# Reset index for easier viewing
df = df.reset_index(drop=True)

,date,home_team,away_team,home_xg,away_xg,home_xthreat,away_xthreat,total_xg_for_match,home_team_prev_xg,away_team_prev_xg,home_team_prev_xthreat,away_team_prev_xthreat
0,2023-06-20 20:53:00+00:00,Iceland,Portugal,0.76,0.86,0.83,2.20,1.62,NaN,NaN,NaN,NaN
1,2023-08-05 07:16:58+00:00,Switzerland,Spain,0.57,3.44,0.41,3.08,4.01,NaN,NaN,NaN,NaN
2,2023-08-05 09:58:30+00:00,Japan,Norway,1.58,0.47,1.65,1.28,2.05,NaN,NaN,NaN,NaN
3,2023-08-06 04:28:25+00:00,Netherlands,South,1.80,1.17,1.43,0.86,2.97,NaN,NaN,NaN,NaN
4,2023-08-06 18:24:15+00:00,Arsenal,Man,0.69,0.96,0.99,1.05,1.65,NaN,NaN,NaN,NaN


In [23]:
df_filtered = df[['total_xg_for_match', 'home_team_prev_xg', 'away_team_prev_xg', 'home_team_prev_xthreat', 'away_team_prev_xthreat']]

In [26]:
df_filtered = df_filtered.dropna()

In [29]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.model_selection import train_test_split

# Features and target variable
X = df_filtered[['home_team_prev_xg', 'away_team_prev_xg', 'home_team_prev_xthreat', 'away_team_prev_xthreat']]
y = df_filtered['total_xg_for_match']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define kernel: ConstantKernel * RBF
kernel = C(1.0, (1e-3, 1e3)) * RBF(1, (1e-2, 1e2))

# Gaussian Process Regression model
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=0)

# Fit model on training data
gpr.fit(X_train, y_train)

# Make predictions on the test set
y_pred, sigma = gpr.predict(X_test, return_std=True)

# Display the test set results
results = pd.DataFrame({
    'actual_xg': y_test,
    'predicted_xg': y_pred,
    'prediction_sigma': sigma
})

print("Test Set Results:")
print(results)


Test Set Results:
     actual_xg  predicted_xg  prediction_sigma
67        4.94      2.660030          1.169483
116       2.53      3.084378          1.152744
115       1.94      2.009339          1.218242
60        2.39      0.770190          2.150992
110       2.61      1.158501          2.072173
40        2.50      0.623304          2.725174
65        1.57      3.847272          1.640508
119       1.69      0.365664          1.589383
38        3.05      1.978760          0.461939
82        3.54      1.199446          0.586979
61        3.79      4.207036          1.164006


In [30]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [31]:
# Initialize Decision Tree Regressor
dtr = DecisionTreeRegressor(random_state=42)

# Fit model on training data
dtr.fit(X_train, y_train)

# Make predictions on the test set
y_pred = dtr.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("R-squared:", r2)

# Display the test set results
results = pd.DataFrame({
    'actual_xg': y_test,
    'predicted_xg': y_pred
})

print("Test Set Results:")
print(results)

Mean Squared Error: 2.524772727272727
R-squared: -1.7728638335723494
Test Set Results:
     actual_xg  predicted_xg
67        4.94          2.85
116       2.53          2.15
115       1.94          4.54
60        2.39          2.15
110       2.61          1.50
40        2.50          4.82
65        1.57          3.33
119       1.69          2.15
38        3.05          0.78
82        3.54          4.61
61        3.79          4.26
